# Phase 5-B2｜Controlled Execution & Atomic Budget Acceptance

**Notebook 版本：`PHASE5B2-EXECUTION-v3`**　｜　Code pin `9f73a56`　｜　分支 `feature/phase5b2-controlled-execution`

`Proposal → Bound Request → Verified Approval → Governance → Atomic Budget Reservation → Controlled Runner → Evaluation → Archive → Budget Settlement`

- **全部是隔離的 SYNTHETIC 環境。** 正式 `RESEARCH_ONLY_REAL` 維持 DISABLED；不執行真實市場回測、不使用正式授權額度、不讀 Holdout、不呼叫券商 API。
- **本版是 PR #13 安全修補後的驗收（Codex P0-1 / P0-2 / P0-3）。** 先前的 `executions/20261108_115819_019db14` 是修補前的結果，保留不覆寫。
- **Layer A**：身分設定好的 SYNTHETIC_STUB backend，不需要 Core；只驗證帳本／證據機制，不是 producer 產出。
- **Layer B**：既有 `ResearchOnlyRunner` 搭配 Core `b16179a`，經由 Orchestrator 跑完整流程（A2 → 已驗證的 binding → bound request），資料是 Core 產生的 **synthetic** development snapshot。
- 三項 P0 會在新的 subprocess 裡用重新推導的重現腳本實際攻擊一次，結果寫入 `security_reproduction.json`。本 notebook **不宣告 Security GO**，安全結論交由 Codex 獨立複審。
- 一致性範圍為 `SINGLE_EXECUTOR_ONLY`：這個 notebook 是唯一的 executor，不宣稱跨 runtime 的原子性。
- 本 notebook 不需要 GitHub token。
- 輸出：`MyDrive/Quant_Research/strategy-research-agent/executions/<YYYYMMDD_HHMMSS>_<pin7>/`，最後寫入 `ARCHIVE_INDEX.json`，所有檔案不可覆寫；並會比對下載的 ZIP 與 Drive 上的封存。

**請在全新的 Colab runtime 從 GitHub 開啟，然後 Run all。**

In [ ]:
# --- 1. 設定（不要修改 pin）---
NOTEBOOK_VERSION = "PHASE5B2-EXECUTION-v3"
NOTEBOOK_PATH    = "notebooks/Phase5B2_Controlled_Execution_Acceptance.ipynb"
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH     = "feature/phase5b2-controlled-execution"
PIN_SHA    = "9f73a568b04656ce6b9523adf1e871f9ebd0fd9f"
CORE_SHA   = "b16179a7cc077c36046574769a32c367fa476cae"
DRIVE_BASE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
AGENT_ROOT = "/content/strategy-research-agent-phase5b2"

import json, os, re, subprocess, sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
assert sys.version_info >= (3, 11), sys.version
CLEAN_RUNTIME = not any(m in sys.modules for m in ("strategy_agent", "market_research_core"))
AGENT_ROOT = Path(AGENT_ROOT)
def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if r.returncode: raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-400:])
    return r.stdout.strip()
ENV = {k: v for k, v in os.environ.items() if k not in ("GH_TOKEN", "GITHUB_TOKEN")}
def py(args, **kw):
    return subprocess.run([sys.executable, *args], cwd=AGENT_ROOT, text=True, capture_output=True, env=ENV, **kw)
print(NOTEBOOK_VERSION, "| pin", PIN_SHA[:7], "| python", sys.version.split()[0], "| clean runtime:", CLEAN_RUNTIME)

In [ ]:
# --- 2. Version Sync ---
assert CLEAN_RUNTIME, "STOP: 此 runtime 已載入 strategy_agent / market_research_core；請 Runtime → Restart session 後 Run all"
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT)); git("checkout", "--quiet", "--detach", PIN_SHA, cwd=AGENT_ROOT)
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", PIN_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0
nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
gh_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', nb.stdout) or [None, None])[1] if nb.returncode == 0 else None
after_pin = git("diff", "--name-only", PIN_SHA, remote_head, cwd=AGENT_ROOT).splitlines() if remote_head != PIN_SHA else []
SYNC = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": gh_version, "github_remote_head": remote_head,
        "checkout_head": head, "pinned_sha": PIN_SHA, "pin_on_branch": on_branch, "working_tree_clean": status == "",
        "files_changed_after_pin": after_pin}
print(json.dumps(SYNC, indent=2))
assert head == PIN_SHA and status == "" and on_branch, "STOP: checkout does not match the pin"
assert gh_version in (None, NOTEBOOK_VERSION), f"STOP: 這是舊 notebook；GitHub 上是 {gh_version}"
code_after_pin = [p for p in after_pin if not p.startswith(("docs/", "notebooks/"))]
assert not code_after_pin, f"STOP: pin 之後 GitHub 上有程式／治理變更 {code_after_pin}；需要新的 pin"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT) + "[test,research]"], check=True)
print("VERSION SYNC: PASS")

In [ ]:
# --- 3. 完整回歸（尚未安裝 Core）：任何非零結束碼都 STOP ---
t = py(["-m", "pytest", "-q", "-p", "no:cacheprovider"], timeout=3600)
REG = {"returncode": t.returncode, "summary": t.stdout.strip().splitlines()[-1] if t.stdout.strip() else "", "core": "absent"}
print(t.stdout[-1200:]); print(REG)
assert t.returncode == 0, "STOP: regression failed"

In [ ]:
# --- 4. 安裝 Core b16179a（唯讀，只用於 Layer B）並跑 B2 suites（junit）---
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                f"market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}"], check=True)
JUNIT = Path("/content/b2_junit.xml")
b = py(["-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_phase5b2_controlled.py",
        "tests/test_phase5b2_runner_core.py", "tests/test_phase5b2_security_remediation.py",
        "tests/test_phase5b2_acceptance.py", "--junitxml", str(JUNIT)], timeout=3600)
B2 = {"returncode": b.returncode, "summary": b.stdout.strip().splitlines()[-1] if b.stdout.strip() else "", "core": CORE_SHA}
print(b.stdout[-1500:]); print(B2)
assert b.returncode == 0, "STOP: B2 suites failed"
assert "skipped" not in B2["summary"], "STOP: Layer B was skipped (Core not active)"
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: tests modified the checkout"

In [ ]:
# --- 5. Drive + B2-01..B2-10 acceptance（全新 subprocess）---
CLOUD = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CLOUD = Path(DRIVE_BASE).is_dir()
except Exception as e:                       # noqa: BLE001
    print("Drive unavailable:", type(e).__name__, e)
OUT_BASE = Path(DRIVE_BASE) / "executions" if CLOUD else Path("/content/executions_LOCAL_CLOUD_PENDING")
OUT = OUT_BASE / f"{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}_{PIN_SHA[:7]}"
assert not OUT.exists(), "STOP: output exists (never overwritten)"
cj = Path("/content/b2_context.json"); cj.write_text(json.dumps({"notebook": NOTEBOOK_VERSION, "version_sync": SYNC,
    "clean_colab_runtime": CLEAN_RUNTIME, "cloud": CLOUD, "python": sys.version.split()[0], "regression": REG, "b2_suites": B2,
    "executor": "this Colab runtime (SINGLE_EXECUTOR_ONLY)"}))
rc = 0 if (REG["returncode"] == 0 and B2["returncode"] == 0) else 1
cmd = ["-m", "strategy_agent.execution.acceptance", "--out-dir", str(OUT), "--junit", str(JUNIT), "--tests-rc", str(rc),
       "--context-json", str(cj)] + (["--cloud"] if CLOUD else [])
r = py(cmd, timeout=3600)
print(r.stdout[-4000:]); print(r.stderr[-2000:])
RES = json.loads(r.stdout[r.stdout.index("{"):])
print("VERDICT:", RES["verdict"])
print("SECURITY REMEDIATION (not a Security GO):", RES.get("security_remediation"))
assert set(RES.get("security_remediation", {}).values()) == {"FIXED_PENDING_INDEPENDENT_REVIEW"},     "STOP: a P0 finding is not fixed / not evidenced in this runtime"
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: acceptance modified the checkout"

In [ ]:
# --- 6. ZIP + 封存一致性驗證（ARCHIVE_INDEX 涵蓋全部檔案；之後不再寫入）---
import zipfile
zp = Path("/content") / (OUT.name + "_phase5b2_execution.zip")
with zipfile.ZipFile(zp, "w") as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p, p.name)
v = py(["-m", "strategy_agent.execution.acceptance", "--out-dir", str(OUT), "--verify-archive", "--zip", str(zp)])
print(v.stdout); assert v.returncode == 0, "STOP: archive / ZIP verification failed"
print((OUT / "run_info.txt").read_text())
try:
    from google.colab import files as F; F.download(str(zp))
except Exception as e:                       # noqa: BLE001
    print("download skipped:", e, "| zip:", zp)